# 🔧 REBUILD — Perakitan Ulang Dataset Berita dari Checkpoint

Notebook ini **merakit ulang** CSV berita dari folder `checkpoint_*` hasil scraping
yang sudah berjalan, **tanpa mengunduh ulang satu artikel pun**.

### Mengapa perlu
| Masalah pada data lama | Diperbaiki di sini |
|---|---|
| **Kontan 0 artikel** — checkpoint berisi ±2 MB data, tetapi CSV final gagal ditulis (0 KB) | Dirakit ulang dari checkpoint ✅ |
| **Dedup `subset='Link'`** membuang artikel yang sah relevan untuk 2 emiten | Dedup `['Link','Kode_Emiten']` ✅ |
| **Pencocokan alias substring** → *false positive* ('pgn' cocok di "pgnXYZ") | Pencocokan **batas kata** ✅ |
| Kriteria relevansi **berbeda-beda** antar scraper | **Kriteria tunggal & seragam** untuk 5 sumber ✅ |

### Yang TIDAK bisa dipulihkan di sini
- **Detik–ITMG** (2023 & 2024 = 0 artikel): artikelnya tidak pernah terunduh akibat bug
  `all_old`, jadi tidak ada di checkpoint → **perlu scraping ulang** dengan
  `scraping_detik_REVISI_v2.ipynb`.
- Folder `checkpoint_idxchannel` (0 KB) & `checkpoint_wartaekonomi` (kosong) diabaikan —
  bukan bagian dari 5 sumber resmi penelitian.

> ⚠️ Notebook ini **tidak menimpa** CSV lama. Keluarannya bernama `*_bersih_v2.csv`.

## 1. Konfigurasi

In [2]:
import re
import shutil
import pandas as pd
from pathlib import Path
from datetime import datetime

# ── Akar proyek (sesuaikan bila folder dipindah) ──
BASE = Path(r'E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
            r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026')
DIR_CHECKPOINT = BASE / '01_scraping_data' / 'berita_sentimen'
DIR_RAW        = BASE / '03_dataset' / 'raw'
DIR_BACKUP     = DIR_RAW / '_backup_sebelum_rebuild'

assert BASE.exists(),           f'Folder proyek tidak ditemukan: {BASE}'
assert DIR_CHECKPOINT.exists(), f'Folder checkpoint tidak ditemukan: {DIR_CHECKPOINT}'
DIR_BACKUP.mkdir(parents=True, exist_ok=True)

# ── Periode penelitian (SATU sumber kebenaran — sinkron Bab 1, 3, 4) ──
START = pd.Timestamp('2022-01-01')
END   = pd.Timestamp('2026-06-30 23:59:59')

# ── Pemetaan folder checkpoint -> label Sumber (5 sumber resmi tesis) ──
# checkpoint_idxchannel (0 KB) & checkpoint_wartaekonomi (kosong) SENGAJA
# tidak dimasukkan: bukan bagian dari lima portal yang ditetapkan di Bab 3.
PETA_SUMBER = {
    'checkpoint_detik'    : 'Detik Finance',
    'checkpoint_antara'   : 'ANTARA News',
    'checkpoint_kontan'   : 'Kontan',
    'checkpoint_bloomberg': 'Bloomberg Technoz',
    'checkpoint_cnbc'     : 'CNBC Indonesia',
}
# Nama file keluaran per sumber
PETA_FILE = {
    'Detik Finance'    : 'berita_detik_bersih_v2.csv',
    'ANTARA News'      : 'berita_antara_bersih_v2.csv',
    'Kontan'           : 'berita_kontan_bersih_v2.csv',
    'Bloomberg Technoz': 'berita_bloomberg_bersih_v2.csv',
    'CNBC Indonesia'   : 'berita_cnbc_bersih_v2.csv',
}
FILE_GABUNGAN = 'berita_semua_sumber_bersih_v2.csv'

URUTAN_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']
KOLOM_STANDAR = ['Sumber', 'Kode_Emiten', 'Keyword', 'Headline',
                 'Tanggal', 'Link', 'Isi_Berita']

print('=' * 72)
print('  KONFIGURASI REBUILD')
print('=' * 72)
print(f'  Checkpoint : {DIR_CHECKPOINT}')
print(f'  Output     : {DIR_RAW}')
print(f'  Periode    : {START:%d %b %Y} s/d {END:%d %b %Y}')
print('\n  Folder checkpoint terdeteksi:')
for d in sorted(DIR_CHECKPOINT.glob('checkpoint_*')):
    n_file = len(list(d.glob('*_checkpoint.csv')))
    ukuran = sum(f.stat().st_size for f in d.glob('*_checkpoint.csv')) / 1024
    status = '-> DIPAKAI' if d.name in PETA_SUMBER else '-> diabaikan (bukan sumber resmi/kosong)'
    print(f'    {d.name:24} {n_file} file  {ukuran:7.0f} KB  {status}')
print('=' * 72)

  KONFIGURASI REBUILD
  Checkpoint : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\01_scraping_data\berita_sentimen
  Output     : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\raw
  Periode    : 01 Jan 2022 s/d 30 Jun 2026

  Folder checkpoint terdeteksi:
    checkpoint_antara        7 file     5842 KB  -> DIPAKAI
    checkpoint_bloomberg     7 file      310 KB  -> DIPAKAI
    checkpoint_cnbc          7 file     2953 KB  -> DIPAKAI
    checkpoint_detik         7 file     3090 KB  -> DIPAKAI
    checkpoint_idxchannel    7 file        0 KB  -> diabaikan (bukan sumber resmi/kosong)
    checkpoint_kontan        7 file     2075 KB  -> DIPAKAI
    checkpoint_wartaekonomi  0 file        0 KB  -> diabaikan (bukan sumber resmi/kosong)


## 2. Kriteria Relevansi Seragam (Batas Kata)

Satu kriteria untuk **kelima sumber** — menggantikan filter yang sebelumnya berbeda-beda
tiap scraper. Keseragaman ini penting agar kriteria inklusi data dapat dipertanggungjawabkan
di Bab 3 (tidak ada sumber yang disaring lebih longgar dari yang lain).

**Aturan penerimaan:** artikel diterima bila (1) menyebut emiten sebagai **kata utuh**,
DAN (2) memuat minimal satu **kata konteks pasar** ATAU emiten disebut di **judul**.

In [3]:
# ── Pencocokan berbatas kata: cegah 'pgn' cocok di "pgnXYZ", 'akr' di "akronim" ──
def _sebut(alias_list, teks):
    for a in alias_list:
        a = str(a).strip()
        if a and re.search(rf'(?<!\w){re.escape(a)}(?!\w)', teks):
            return True
    return False

EMITEN_ALIAS = {
    'ADRO': ['adro', 'adaro energy', 'adaro', 'alamtri'],
    'ADMR': ['admr', 'adaro minerals'],
    'ITMG': ['itmg', 'indo tambangraya', 'tambangraya'],
    'PTBA': ['ptba', 'bukit asam'],
    'MEDC': ['medc', 'medco'],
    'AKRA': ['akra', 'akr corporindo'],
    'PGAS': ['pgas', 'perusahaan gas negara', 'pgn'],
}

KATA_KONTEKS_SAHAM = [
    'saham', 'bursa', 'bei', 'idx', 'ihsg', 'emiten', 'investor',
    'laba', 'rugi', 'pendapatan', 'revenue', 'kinerja', 'keuangan',
    'kuartal', 'dividen', 'buyback', 'merger', 'akuisisi',
    'rights issue', 'right issue', 'ipo', 'target harga', 'rekomendasi',
    'analis', 'sekuritas', 'valuasi', 'harga saham', 'koreksi',
    'menguat', 'melemah', 'melesat', 'produksi', 'batubara', 'batu bara',
    'minyak', 'gas', 'lifting', 'laba bersih', 'net profit',
    'pasar modal', 'obligasi', 'rupst', 'capex', 'ekspor', 'penjualan',
]

def alasan_tolak(kode, headline, isi):
    """Return None bila artikel DITERIMA, atau string alasan bila ditolak."""
    hl   = str(headline).lower()
    teks = (hl + ' ' + str(isi)).lower()
    alias = EMITEN_ALIAS.get(kode, [kode.lower()])

    if len(str(isi).strip()) < 100:
        return 'isi_terlalu_pendek'
    if not _sebut(alias, teks):
        return 'tidak_sebut_emiten'          # <- penyaring false positive utama
    n_konteks = sum(1 for k in KATA_KONTEKS_SAHAM if k in teks)
    if n_konteks >= 1 or _sebut(alias, hl):
        return None
    return 'konteks_pasar_lemah'

# Uji cepat kebenaran logika
print('Uji batas kata:')
print('  "pgnxyz adalah akronim"      ->', _sebut(['pgn'], 'pgnxyz adalah akronim'), '(harus False)')
print('  "laba PGN naik 12%"          ->', _sebut(['pgn'], 'laba pgn naik 12%'),      '(harus True)')
print('  "akronim baru"               ->', _sebut(['akra'], 'akronim baru'),          '(harus False)')
print('\nKriteria relevansi seragam siap.')

Uji batas kata:
  "pgnxyz adalah akronim"      -> False (harus False)
  "laba PGN naik 12%"          -> True (harus True)
  "akronim baru"               -> False (harus False)

Kriteria relevansi seragam siap.


## 3. Muat Seluruh Checkpoint

In [4]:
potongan, log_muat = [], []

for folder, sumber in PETA_SUMBER.items():
    d = DIR_CHECKPOINT / folder
    if not d.exists():
        log_muat.append(f'  ⚠️  {folder:24} TIDAK ADA — dilewati'); continue

    n_sumber = 0
    for f in sorted(d.glob('*_checkpoint.csv')):
        if f.stat().st_size == 0:
            continue                       # checkpoint kosong (mis. idxchannel)
        try:
            df = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)
        except Exception as e:
            log_muat.append(f'  ❌ {folder}/{f.name}: gagal dibaca ({str(e)[:40]})'); continue
        if len(df) == 0:
            continue

        # Kode_Emiten: pakai kolom bila ada, jika tidak ambil dari nama file
        if 'Kode_Emiten' not in df.columns:
            df['Kode_Emiten'] = f.name.split('_')[0].upper()
        df['Sumber'] = sumber              # label sumber SELALU dari folder (konsisten)

        for kol in KOLOM_STANDAR:
            if kol not in df.columns:
                df[kol] = ''
        potongan.append(df[KOLOM_STANDAR].copy())
        n_sumber += len(df)

    log_muat.append(f'  ✅ {sumber:20} {n_sumber:>6,} baris mentah dari {folder}')

print('=' * 72)
print('  MEMUAT CHECKPOINT')
print('=' * 72)
print('\n'.join(log_muat))

df_mentah = pd.concat(potongan, ignore_index=True)
df_mentah['Tanggal'] = pd.to_datetime(df_mentah['Tanggal'], errors='coerce')

print('=' * 72)
print(f'  TOTAL BARIS MENTAH : {len(df_mentah):,}')
print(f'  Tanggal gagal parse: {df_mentah["Tanggal"].isna().sum():,}')
print('=' * 72)

  MEMUAT CHECKPOINT
  ✅ Detik Finance           960 baris mentah dari checkpoint_detik
  ✅ ANTARA News           1,789 baris mentah dari checkpoint_antara
  ✅ Kontan                  660 baris mentah dari checkpoint_kontan
  ✅ Bloomberg Technoz       233 baris mentah dari checkpoint_bloomberg
  ✅ CNBC Indonesia        1,013 baris mentah dari checkpoint_cnbc
  TOTAL BARIS MENTAH : 4,655
  Tanggal gagal parse: 0


## 4. Penyaringan: Periode, Relevansi, Deduplikasi

In [5]:
df = df_mentah.copy()
jejak = {'0. Mentah dari checkpoint': len(df)}

# ── (a) Tanggal valid ──
df = df.dropna(subset=['Tanggal'])
jejak['1. Tanggal valid'] = len(df)

# ── (b) Dalam periode penelitian ──
df = df[(df['Tanggal'] >= START) & (df['Tanggal'] <= END)]
jejak['2. Dalam periode 2022-2026'] = len(df)

# ── (c) Link valid ──
df = df[df['Link'].astype(str).str.startswith('http')]
jejak['3. Link valid'] = len(df)

# ── (d) Relevansi seragam (batas kata) ──
df['_alasan'] = df.apply(
    lambda r: alasan_tolak(r['Kode_Emiten'], r['Headline'], r['Isi_Berita']), axis=1)
ditolak = df[df['_alasan'].notna()]
df = df[df['_alasan'].isna()].drop(columns='_alasan')
jejak['4. Lolos relevansi'] = len(df)

# ── (e) Dedup pada PASANGAN (Link, Kode_Emiten) ──
# Artikel yang sah relevan untuk DUA emiten tetap disimpan dua kali dengan
# Kode_Emiten berbeda — inilah perbaikan atas dedup lama subset='Link'.
n_seb = len(df)
df = df.drop_duplicates(subset=['Link', 'Kode_Emiten'], keep='first')
jejak['5. Setelah dedup (Link+Emiten)'] = len(df)

df = df.sort_values(['Tanggal', 'Kode_Emiten']).reset_index(drop=True)

print('=' * 72)
print('  JEJAK PENYARINGAN')
print('=' * 72)
sebelumnya = None
for tahap, n in jejak.items():
    delta = f'  (-{sebelumnya - n:,})' if sebelumnya is not None and sebelumnya != n else ''
    print(f'  {tahap:34} : {n:>7,}{delta}')
    sebelumnya = n

print('\n  Rincian artikel ditolak (per alasan):')
if len(ditolak):
    for alasan, n in ditolak['_alasan'].value_counts().items():
        print(f'    {alasan:24} : {n:>6,}')
    print('\n  Ditolak per sumber x emiten (mayoritas = false positive alias):')
    print(pd.crosstab(ditolak['Sumber'], ditolak['Kode_Emiten'])
            .reindex(columns=URUTAN_EMITEN, fill_value=0).to_string())
else:
    print('    (tidak ada)')
print('=' * 72)

  JEJAK PENYARINGAN
  0. Mentah dari checkpoint          :   4,655
  1. Tanggal valid                   :   4,655
  2. Dalam periode 2022-2026         :   4,655
  3. Link valid                      :   4,655
  4. Lolos relevansi                 :   4,474  (-181)
  5. Setelah dedup (Link+Emiten)     :   4,474

  Rincian artikel ditolak (per alasan):
    tidak_sebut_emiten       :    174
    konteks_pasar_lemah      :      7

  Ditolak per sumber x emiten (mayoritas = false positive alias):
Kode_Emiten        ADRO  ADMR  ITMG  PTBA  MEDC  AKRA  PGAS
Sumber                                                     
ANTARA News           2     1    12     0     3     7     0
Bloomberg Technoz    22    20    20    20    19    18    18
CNBC Indonesia        1     0     0     1     0     2     1
Detik Finance         1     0     0     2     4     4     1
Kontan                0     0     1     1     0     0     0


## 5. Simpan Hasil (CSV lama TIDAK ditimpa)

In [6]:
# ── Backup CSV lama sebagai arsip (sekali saja) ──
for f in DIR_RAW.glob('berita_*.csv'):
    tujuan = DIR_BACKUP / f.name
    if not tujuan.exists():
        shutil.copy2(f, tujuan)
print(f'  📦 Arsip CSV lama -> {DIR_BACKUP.name}/ ({len(list(DIR_BACKUP.glob("*.csv")))} file)\n')

# ── Simpan per sumber ──
print('=' * 72)
print('  MENYIMPAN HASIL BERSIH')
print('=' * 72)
for sumber, nama_file in PETA_FILE.items():
    sub = df[df['Sumber'] == sumber].copy()
    sub['Tanggal'] = sub['Tanggal'].dt.strftime('%Y-%m-%d %H:%M')
    sub[KOLOM_STANDAR].to_csv(DIR_RAW / nama_file, index=False, encoding='utf-8-sig')
    print(f'  ✅ {nama_file:34} : {len(sub):>6,} artikel')

# ── Simpan gabungan (dipakai PREP_02) ──
gab = df.copy()
gab['Tanggal'] = gab['Tanggal'].dt.strftime('%Y-%m-%d %H:%M')
gab[KOLOM_STANDAR].to_csv(DIR_RAW / FILE_GABUNGAN, index=False, encoding='utf-8-sig')
print(f'\n  💾 {FILE_GABUNGAN:34} : {len(gab):>6,} artikel  <- input PREP_02')
print('=' * 72)

  📦 Arsip CSV lama -> _backup_sebelum_rebuild/ (9 file)

  MENYIMPAN HASIL BERSIH
  ✅ berita_detik_bersih_v2.csv         :    948 artikel
  ✅ berita_antara_bersih_v2.csv        :  1,764 artikel
  ✅ berita_kontan_bersih_v2.csv        :    658 artikel
  ✅ berita_bloomberg_bersih_v2.csv     :     96 artikel
  ✅ berita_cnbc_bersih_v2.csv          :  1,008 artikel

  💾 berita_semua_sumber_bersih_v2.csv  :  4,474 artikel  <- input PREP_02


## 6. Rekap Final (→ Tabel 4.x Bab 4)

In [7]:
print('=' * 78)
print('  REKAP DATASET BERITA BERSIH — 5 SUMBER')
print('=' * 78)
print(f'  Total artikel : {len(df):,}')
print(f'  Periode       : {df["Tanggal"].min():%d %b %Y}  s/d  {df["Tanggal"].max():%d %b %Y}')
print(f'  Artikel unik  : {df["Link"].nunique():,} (selisih = artikel multi-emiten)')

print('\n  ── Distribusi Sumber x Emiten (bahan Tabel 4.4) ──')
tab = pd.crosstab(df['Sumber'], df['Kode_Emiten'],
                  margins=True, margins_name='TOTAL')
tab = tab.reindex(columns=URUTAN_EMITEN + ['TOTAL'], fill_value=0)
print(tab.to_string())

print('\n  ── Distribusi per Tahun x Sumber ──')
print(pd.crosstab(df['Tanggal'].dt.year, df['Sumber'],
                  margins=True, margins_name='TOTAL').to_string())

# Indikator presisi penyaringan
prop = df.apply(lambda r: _sebut(EMITEN_ALIAS.get(r['Kode_Emiten'], []),
                                 str(r['Headline']).lower()), axis=1).mean() * 100
print(f'\n  Proporsi emiten disebut di JUDUL : {prop:.0f}%  (makin tinggi makin presisi)')

# Simpan tabel rekap untuk dokumen tesis
DIR_TABEL = BASE / '06_dokumen_tesis' / 'tabel'
DIR_TABEL.mkdir(parents=True, exist_ok=True)
tab.to_csv(DIR_TABEL / 'tabel_distribusi_berita_sumber_emiten.csv', encoding='utf-8-sig')
print(f'\n  💾 tabel_distribusi_berita_sumber_emiten.csv tersimpan di 06_dokumen_tesis/tabel/')
print('=' * 78)

  REKAP DATASET BERITA BERSIH — 5 SUMBER
  Total artikel : 4,474
  Periode       : 01 Jan 2022  s/d  24 Jun 2026
  Artikel unik  : 4,085 (selisih = artikel multi-emiten)

  ── Distribusi Sumber x Emiten (bahan Tabel 4.4) ──
Kode_Emiten        ADRO  ADMR  ITMG  PTBA  MEDC  AKRA  PGAS  TOTAL
Sumber                                                            
ANTARA News         258    26    47   460   293    51   629   1764
Bloomberg Technoz    27     8    10    25    11     4    11     96
CNBC Indonesia      268    38    73   276    96    44   213   1008
Detik Finance       195    47    44   188   196    38   240    948
Kontan              118    70    89    95    84    95   107    658
TOTAL               866   189   263  1044   680   232  1200   4474

  ── Distribusi per Tahun x Sumber ──
Sumber   ANTARA News  Bloomberg Technoz  CNBC Indonesia  Detik Finance  Kontan  TOTAL
Tanggal                                                                              
2022             277         

## 7. Validasi Mutu Data

In [8]:
print('=' * 72)
print('  VALIDASI MUTU DATA')
print('=' * 72)
cek = {
    'Duplikat (Link, Kode_Emiten)' : df.duplicated(subset=['Link','Kode_Emiten']).sum(),
    'Tanggal kosong'               : df['Tanggal'].isna().sum(),
    'Di luar periode penelitian'   : ((df['Tanggal'] < START) | (df['Tanggal'] > END)).sum(),
    'Isi_Berita < 100 karakter'    : (df['Isi_Berita'].fillna('').str.len() < 100).sum(),
    'Headline kosong'              : (df['Headline'].fillna('').str.strip() == '').sum(),
    'Emiten di luar 7 kandidat'    : (~df['Kode_Emiten'].isin(URUTAN_EMITEN)).sum(),
}
for k, v in cek.items():
    print(f'  {"✅" if v == 0 else "⚠️ "} {k:32}: {v:,}')

kosong = [e for e in URUTAN_EMITEN if (df['Kode_Emiten'] == e).sum() == 0]
print(f'\n  {"✅" if not kosong else "⚠️ "} Emiten tanpa artikel sama sekali : '
      f'{kosong if kosong else "tidak ada"}')

# Sel-per-sel: kombinasi sumber x emiten yang NOL (perlu dicurigai)
nol = [(s, e) for s in PETA_SUMBER.values() for e in URUTAN_EMITEN
       if ((df['Sumber'] == s) & (df['Kode_Emiten'] == e)).sum() == 0]
print(f'  {"✅" if not nol else "⚠️ "} Kombinasi sumber x emiten = 0     : '
      f'{nol if nol else "tidak ada"}')

print('\n' + '=' * 72)
print('  LANGKAH SELANJUTNYA')
print('=' * 72)
print('  1. Scraping ulang Detik-ITMG  -> scraping_detik_REVISI_v2.ipynb')
print('     (2023 & 2024 = 0 artikel akibat bug all_old; tidak ada di checkpoint)')
print('  2. PREP_02 (IndoBERT) -> arahkan input ke:', FILE_GABUNGAN)
print('     dan HAPUS pemakaian file *_tambahan_negatif.csv (pendekatan ditinggalkan)')
print('  3. PREP_01 -> ubah tanggal unduhan yfinance ke 2021-11-01 (buffer warm-up),')
print('     filter final Date >= 2022-01-01  [penyebab dataset hanya 746 hari]')
print('  4. PREP_03 v2 -> MODEL_xgboost_v3 -> MODEL_lstm_v4')
print('=' * 72)

  VALIDASI MUTU DATA
  ✅ Duplikat (Link, Kode_Emiten)    : 0
  ✅ Tanggal kosong                  : 0
  ✅ Di luar periode penelitian      : 0
  ✅ Isi_Berita < 100 karakter       : 0
  ✅ Headline kosong                 : 0
  ✅ Emiten di luar 7 kandidat       : 0

  ✅ Emiten tanpa artikel sama sekali : tidak ada
  ✅ Kombinasi sumber x emiten = 0     : tidak ada

  LANGKAH SELANJUTNYA
  1. Scraping ulang Detik-ITMG  -> scraping_detik_REVISI_v2.ipynb
     (2023 & 2024 = 0 artikel akibat bug all_old; tidak ada di checkpoint)
  2. PREP_02 (IndoBERT) -> arahkan input ke: berita_semua_sumber_bersih_v2.csv
     dan HAPUS pemakaian file *_tambahan_negatif.csv (pendekatan ditinggalkan)
  3. PREP_01 -> ubah tanggal unduhan yfinance ke 2021-11-01 (buffer warm-up),
     filter final Date >= 2022-01-01  [penyebab dataset hanya 746 hari]
  4. PREP_03 v2 -> MODEL_xgboost_v3 -> MODEL_lstm_v4
